# Game Industry Data Analysis for "Streamchik" Online Store

## Navigation

- [Step 1 Data Exploration](#step-1-data-exploration)
- [Step 2 Data Preprocessing](#step-2-data-preprocessing)
  - [Renaming Columns](#renaming-columns)
  - [Changing Data Types](#changing-data-types)
  - [Handling 'tbd' Values in the user_score column](#handling-tbd-values-in-the-user_score-column)
  - [Handling Missing Values in the year_of_release column](#handling-missing-values-in-the-year_of_release-column)
  - [Handling Missing Values in the critic_score Column](#handling-missing-values-in-the-critic_score-column)
  - [Handling Missing Values in the user_score Column](#handling-missing-values-in-the-user_score-column)
  - [Handling Missing Values in the rating Column](#handling-missing-values-in-the-rating-column)
  - [Calculating Total Sales](#calculating-total-sales)
  - [Checking for Implicit Duplicates](#checking-for-implicit-duplicates)
  - [Data Preprocessing Conclusion](#data-preprocessing-conclusion)
- [Step 3 Exploratory Data Analysis](#step-3-exploratory-data-analysis)
  - [Sales by Platform and Their Dynamics](#sales-by-platform-and-their-dynamics)
  - [Selecting Relevant Data](#selecting-relevant-data)
  - [Selecting Potentially Profitable Platforms](#selecting-potentially-profitable-platforms)
  - [Box Plot of Global Sales by Platform](#box-plot-of-global-sales-by-platform)
  - [Impact of Reviews on Sales Within a Popular Platform](#impact-of-reviews-on-sales-within-a-popular-platform)
  - [Correlation of Conclusions with Sales on Other Platforms](#correlation-of-conclusions-with-sales-on-other-platforms)
  - [Impact of Genre on Game Sales](#impact-of-genre-on-game-sales)
  - [Exploratory Analysis Conclusion](#exploratory-analysis-conclusion)
- [Step 4 Create a User Profile for Each Region](#step-4-create-a-user-profile-for-each-region)
  - [North America Region](#north-america-region)
  - [Europe Region](#europe-region)
  - [Japan Region](#japan-region)
  - [Conclusion](#conclusion)
- [Step 5 Hypothesis Testing](#step-5-hypothesis-testing)
  - [Mean user ratings for Xbox One and PC platforms are equal](#mean-user-ratings-for-xbox-one-and-pc-platforms-are-equal)
  - [Mean user ratings for Action and Sports genres are different.](#mean-user-ratings-for-action-and-sports-genres-are-different)
- [Overall Conclusion](#overall-conclusion)
  - [Summary of Work Done](#summary-of-work-done)
  - [2017 Forecast](#2017-forecast)


<a id="step-1-data-exploration"></a>
## Step 1 Data Exploration

In [ ]:
# import all necessary modules
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import seaborn as sns

In [ ]:
data = pd.read_csv('/datasets/games.csv')
# read the file

In [ ]:
data.head(10)
# display first 10 rows

In [ ]:
data.info()
# examine general information about the dataframe

**Conclusion:** The info method results show that the table contains empty values and some columns have inappropriate data types. Column names should also be converted to snake_case style.

[Back to navigation](#navigation)


<a id="step-2-data-preprocessing"></a>
## Step 2 Data Preprocessing

<a id="renaming-columns"></a>
### Renaming Columns

In [ ]:
data.columns = data.columns.str.lower()
# convert column names to lowercase

In [ ]:
data.columns
# verify the changes

<a id="changing-data-types"></a>
### Changing Data Types

In [ ]:
data.dtypes
# examine data types in the columns

The data type in the 'year_of_release' column needs to be changed to int (release year is an integer), and 'user_score' to float (user rating takes numeric values from 0 to 10).

In [ ]:
data['year_of_release'] = data['year_of_release'].astype('Int16')
# convert year to nullable integer type that supports NaN values

<a id="handling-tbd-values-in-the-user_score-column"></a>
### Handling 'tbd' Values in the user_score column

Before changing the data type in the 'user_score' column, we need to clarify the 'tbd' value (to be determined). Let's count the number of rows with this value.

In [ ]:
len(data[data['user_score'] == 'tbd'])

In [ ]:
data[data['user_score'] == 'tbd']['year_of_release'].value_counts()
# look at the distribution of 'tbd' values by year

The rating may indeed be undetermined for later years 2014-2016, but we see that most ratings are undefined for 2008-2011, which may indicate outdated data—these games have likely received user ratings over the past 5-8 years. In this case, 'tbd' values can be replaced with NaN since they are essentially no different from missing data.

In [ ]:
data['user_score'] = pd.to_numeric(data['user_score'], errors='coerce')
# convert values in 'user_score' column to numeric type, replacing 'tbd' with NaN

In [ ]:
data.dtypes
# verify the result

<a id="handling-missing-values-in-the-year_of_release-column"></a>
### Handling Missing Values in the year_of_release column

In [ ]:
data.isna().sum()
# examine the number of missing values in all columns

In [ ]:
data[data['name'].isna()]
# examine games with empty values in the name column

These games have many missing values in columns and low sales (especially the second game), so these rows can be safely removed from the dataframe. Note that these same games have empty values in the "genre" column, so removing these rows will also eliminate those missing values.
These missing values may have arisen due to unsuccessful game launches.

In [ ]:
data.dropna(subset=['name'], inplace=True)
# remove rows with missing values

In [ ]:
data.query('year_of_release.isna()')['name'].unique()
# examine game names with missing values in the 'year_of_release' column

From the unique names, we can see that for some games the year is specified in the game name, e.g.: Madden NFL 2004, Major League Baseball 2K6, MLB SlugFest 20-03. Let's write a function that extracts the release year from the game name.

In [ ]:
def return_year(name):
    for i in range(2000, 2017):
        if str(i) in name:
            return i

    for i in range(0, 17):
        if f'2K{i}' in name:
            return 2000 + i
        elif f'20-0{i}' in name:
            return 2000 + i
    return None

In [ ]:
data['year_of_release'] = data['year_of_release'].fillna(data.query('year_of_release.isna()')['name'].apply(return_year))

In [ ]:
data['year_of_release'].isna().sum()
# verify the number of missing values after processing

In [ ]:
data[data['year_of_release'].isna()].isna().sum()
# examine missing values in other columns in rows with missing release year

We managed to fill 21 missing values in the 'year_of_release' column for games where the release year was specified in the name. Filling the remaining gaps with median or mode could distort results; however, I verified that for most games with missing release year, critic and user scores are known, which may be useful when researching these parameters, so I decided to leave these gaps. Missing values in 'year_of_release' could have occurred if
the year was duplicated in the name (cases we handled) or due to data collection error.

<a id="handling-missing-values-in-the-critic_score-column"></a>
### Handling Missing Values in the critic_score Column

The 'critic_score' column has more than half of values missing. To figure out how to handle them, let's examine how missing values are distributed by year, platform, and genre to find or disprove a pattern.

In [ ]:
nan_counts = (
    data[data['critic_score'].isna()]['year_of_release']
    .value_counts()
    .sort_index()
)
nan_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of missing values in 'critic_score' by release year")
plt.xlabel("Number of missing values")
plt.ylabel("Release year")
plt.show()
# look at the distribution of missing values in 'critic_score' by release year

In [ ]:
game_counts = (
    data['year_of_release']
    .value_counts(dropna=False)
    .sort_index()
)
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by release year")
plt.xlabel("Number of released games")
plt.ylabel("Release year")
plt.show()
# look at the distribution of total released games by release year

In [ ]:
ratio = (
        data[data['critic_score'].isna()]['year_of_release']
        .value_counts()
        /
        data['year_of_release']
        .value_counts()
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'critic_score' by release year")
plt.xlabel("Number of missing values")
plt.ylabel("Release year")
plt.show()

The last chart shows that before 2000, 100% or nearly 100% of games had no critic ratings; this may be because the industry was just developing. The chart of total games by year shows that the number of games in 1980-2000 is many times smaller than in subsequent years. The proportion of missing values is lowest for 2001-2005, which may be linked to the growth of the game industry; however, the total games distribution chart shows that the number of games in those years does not reach 1000, possibly due to the relatively small number and age of these games—they largely received critic ratings. We are most interested in the latest games from 2006 to 2016 inclusive; the game release chart shows a peak in new game releases during these years, with the proportion of missing values in this group varying from 30% to 60%. Missing values in this group should be filled.

In [ ]:
nan_counts = (
    data[data['critic_score'].isna()]['platform']
    .value_counts()
)
nan_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of missing values in 'critic_score' by platform")
plt.xlabel("Number of missing values")
plt.ylabel("Platform")
plt.show()
# look at the distribution of missing values in 'critic_score' by platform

In [ ]:
game_counts = (
    data['platform']
    .value_counts(dropna=False)
)
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by platform")
plt.xlabel("Number of released games")
plt.ylabel("Platform")
plt.show()
# look at the distribution of total released games by platform

In [ ]:
ratio = (
        data[data['critic_score'].isna()]['platform']
        .value_counts()
        /
        data['platform']
        .value_counts()
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'critic_score' by platform")
plt.xlabel("Number of missing values")
plt.ylabel("Platform")
plt.show()

There are many platforms with 100% missing values: 2600, 3DO, GB, GEN, GG, N64, NES, NG, PCFX, SAT, SCD, SNES, TG16, WS. These platforms share the fact that they are hardware platforms for video games from the late 1970s to early 2000s. Games on these platforms are likely from before 2000, consistent with the release year chart. Other proportions range from 20% to 80%.

In [ ]:
nan_counts = (
    data[data['critic_score'].isna()]['genre']
    .value_counts()
)
nan_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of missing values in 'critic_score' by genre")
plt.xlabel("Number of missing values")
plt.ylabel("Genre")
plt.show()
# look at the distribution of missing values in 'critic_score' by genre

In [ ]:
game_counts = (
    data['genre']
    .value_counts()
)
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by genre")
plt.xlabel("Number of released games")
plt.ylabel("Genre")
plt.show()
# look at the distribution of missing values in 'critic_score' by genre

In [ ]:
ratio = (
        data[data['critic_score'].isna()]['genre']
        .value_counts()
        /
        data['genre']
        .value_counts()
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'critic_score' by genre")
plt.xlabel("Number of missing values")
plt.ylabel("Genre")
plt.show()

The proportion of missing values chart shows the highest proportion for genres such as adventure, misc, puzzle, and simulation. The total games by genre chart shows that all these genres (except misc) rank last or near-last in quantity. The misc genre means "other", so the number of games in this group is larger as it includes games of various genres that did not fit a specific category. Thus, we can say that the highest proportion of missing values is in less popular genres that did not attract critic attention.

Based on all the information about the distribution of missing values by release year, platform, and genre, we can develop a filling algorithm. We will fill only the relevant years from 2006 to 2016, excluding old hardware platforms (2600, 3DO, GB, etc.) from filling; in all other cases we will fill gaps with the median by genre for greater robustness to outliers.

In [ ]:
old_platforms = ['2600', '3DO', 'GB', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PCFX', 'SAT', 'SCD', 'SNES', 'TG16', 'WS'] # # create a list of obsolete platforms
years = range(2006, 2017) # create a years variable containing the range from 2006 to 2016 inclusive

In [ ]:
medians = (data
                .query('platform not in @old_platforms and year_of_release in @years')
                .groupby('genre')['critic_score'].median())
# create a variable with critic score medians by genre for relevant years and platforms

In [ ]:
mask = (
    data['critic_score'].isna()
    & ~data['platform'].isin(old_platforms)
    & data['year_of_release'].isin(years)
)
# create a mask for rows where 'critic_score' is empty and year/platform conditions are met

In [ ]:
data.loc[mask, 'critic_score'] = data.loc[mask, 'genre'].map(medians)
# fill critic_score values with medians by genre

In [ ]:
data['critic_score'].isna().sum()
# verify the number of missing values after processing

<a id="handling-missing-values-in-the-user_score-column"></a>
### Handling Missing Values in the user_score Column

The user_score column has even more missing values than critic_score, since when changing the data type we replaced 'tbd' with NaN—essentially a missing value, as some games have been waiting for ratings for several years. So handling missing values in user_score will follow the same approach as critic_score. First, let's examine the distribution of missing values by year, platform, and genre.

In [ ]:
nan_counts = (
    data[data['user_score'].isna()]['year_of_release']
    .value_counts()
    .sort_index()
)
nan_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of missing values in 'user_score' by release year")
plt.xlabel("Number of missing values")
plt.ylabel("Release year")
plt.show()
# look at the distribution of missing values in 'user_score' by release year

In [ ]:
game_counts = (
    data['year_of_release']
    .value_counts(dropna=False)
    .sort_index()
)
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by release year")
plt.xlabel("Number of released games")
plt.ylabel("Release year")
plt.show()
# look at the distribution of total released games by release year

In [ ]:
ratio = (
        data[data['user_score'].isna()]['year_of_release']
        .value_counts()
        /
        data['year_of_release']
        .value_counts()
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'user_score' by release year")
plt.xlabel("Number of missing values")
plt.ylabel("Release year")
plt.show()

As expected, the proportion of missing values in user_score is very similar to that in critic_score. However, for later years starting from 2000, this column typically has 35–65% missing values, slightly higher than critic_score due to additional gaps from replacing 'tbd' with NaN. For filling, we will likewise consider years 2006 to 2016 inclusive.

In [ ]:
ratio = (
        data[data['user_score'].isna()]['platform']
        .value_counts()
        /
        data['platform']
        .value_counts()
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'user_score' by platform")
plt.xlabel("Number of missing values")
plt.ylabel("Platform")
plt.show()

In [ ]:
nan_counts = (
    data[data['user_score'].isna()]['platform']
    .value_counts(dropna=False)
)
nan_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of missing values in 'user_score' by platform")
plt.xlabel("Number of missing values")
plt.ylabel("Platform")
plt.show()
# look at the distribution of missing values in 'critic_score' by platform

In [ ]:
game_counts = (
    data['platform']
    .value_counts()
)
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by platform")
plt.xlabel("Number of released games")
plt.ylabel("Platform")
plt.show()
# look at the distribution of total released games by platform

In [ ]:
ratio = (
        data[data['user_score'].isna()]['platform']
        .value_counts(dropna=False)
        /
        data['platform']
        .value_counts(dropna=False)
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'user_score' by platform")
plt.xlabel("Number of missing values")
plt.ylabel("Platform")
plt.show()

In the resulting distribution, the same old platforms have 100% missing values. The complete absence of ratings for games on these platforms indicates they are obsolete and unused. The proportion of missing values on other platforms ranges from 20% to 85%.

In [ ]:
nan_counts = (
    data[data['user_score'].isna()]['genre']
    .value_counts()
)
nan_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of missing values in 'user_score' by genre")
plt.xlabel("Number of missing values")
plt.ylabel("Genre")
plt.show()
# look at the distribution of missing values in 'user_score' by genre

In [ ]:
game_counts = (
    data['genre']
    .value_counts()
)
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by genre")
plt.xlabel("Number of released games")
plt.ylabel("Genre")
plt.show()
# look at the distribution of missing values in 'critic_score' by genre

In [ ]:
ratio = (
        data[data['user_score'].isna()]['genre']
        .value_counts(dropna=False)
        /
        data['genre']
        .value_counts(dropna=False)
) # calculate the proportion of missing values in total games for each release year
ratio.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of proportion of missing values in 'user_score' by genre")
plt.xlabel("Number of missing values")
plt.ylabel("Genre")
plt.show()

The distribution shows that the most missing values are in adventure, misc, puzzle, and simulation genres—the same genres identified in the critic_score missing-value analysis. However, the percentage of missing values here is somewhat higher, which is again associated with the larger number of gaps in this column. Statistics for both columns suggest these game genres are not very popular with users and, accordingly, with critics either. However, we cannot rule out the possibility that the data were collected incorrectly, leading to the large number of gaps; in any case, they should be sent for review.

The investigation of missing values in user_score shows similarity to critic_score distribution, as rows with gaps likely overlapped, but user_score had about 1000 more missing values. Thus, we will fill missing values in user_score with the median by genre for rows from 2006 to 2016 inclusive, excluding obsolete platforms.

In [ ]:
medians = data.groupby('genre')['user_score'].median()
medians

In [ ]:
mask = (
    data['user_score'].isna()
    & ~data['platform'].isin(old_platforms)
    & data['year_of_release'].isin(years)
)
# create a mask for rows where 'user_score' is empty and year/platform conditions are met

In [ ]:
data.loc[mask, 'user_score'] = data.loc[mask, 'genre'].map(medians)
# fill missing values with median by genre

In [ ]:
data['user_score'].isna().sum()
# verify the result of missing values handling

<a id="handling-missing-values-in-the-rating-column"></a>
### Handling Missing Values in the rating Column

In [ ]:
data['rating'].unique()
# examine unique values in the rating column

Among the unique values there is the 'K-A' (kids to adults) rating, which is not in the main ESRB categories; it is replaced by 'E' (everyone 6+). The 'RP' value means the game has not yet received its rating; these values can be removed from the data.

In [ ]:
data.loc[data['rating'] == 'K-A', 'rating'] = 'E'
# replace 'K-A' values with 'E'

In [ ]:
data = data[data['rating'] != 'RP']

In [ ]:
data['rating'].isna().sum()

The 'rating' column has 6764 missing values; however, we cannot fill them with median as in 'user_score' and 'critic_score' since the values are not numeric. We will leave the missing values in this column unchanged.

<a id="calculating-total-sales"></a>
### Calculating Total Sales

In [ ]:
data['total_sales'] = data.loc[:, 'na_sales':'other_sales'].sum(axis=1)
# create a column with total sales across all regions

<a id="checking-for-implicit-duplicates"></a>
### Checking for Implicit Duplicates

Let's check for implicit duplicates by columns 'name', 'platform', 'year_of_release'

In [ ]:
data.duplicated(subset=['name', 'platform']).sum()
# find the number of implicit duplicates

In [ ]:
data[data.duplicated(subset=['name', 'platform'])]
# examine implicit duplicates

In [ ]:
data.query('name == "Need for Speed: Most Wanted" and platform == "X360"')
# check duplicates for the game 'Need for Speed: Most Wanted'

In [ ]:
data.query('name == "Need for Speed: Most Wanted" and platform == "PC"')
# check duplicates for the game 'Need for Speed: Most Wanted'

I looked into Need for Speed: Most Wanted and found it was actually released in 2005 and 2012, so games on the same platforms but in different years cannot be considered duplicates; we will leave these rows unchanged.

In [ ]:
data.query('name == "Sonic the Hedgehog" and platform == "PS3"')
# check duplicates for the game 'Sonic the Hedgehog'

In this case, the row with 2006 is more complete as it has more sales data; we will keep it in the data.

In [ ]:
data.query('name == "Madden NFL 13" and platform == "PS3"')
# check duplicates for the game 'Madden NFL 13'

Similarly, we will keep row 604 as it has more data

In [ ]:
mask_nfs = data['name'] == 'Need for Speed: Most Wanted' 
# create a mask for Need For Speed: Most Wanted
mask_first = ~data.duplicated(subset=['name','platform'], keep='first') 
# create a mask for first duplicates by name and platform
data = data[mask_nfs | mask_first].reset_index(drop=True)
# overwrite data to keep either Need For Speed: Most Wanted or the first duplicate for other games
# this approach preserves rows for 'Need for Speed', which had no duplicates

<a id="data-preprocessing-conclusion"></a>
### Data Preprocessing Conclusion


During data preprocessing I:
- converted dataframe column names to snake_case
- handled `tbd` (to be determined) values in `user_score` column by replacing them with `NaN`
- changed the data type in `year_of_release` column to nullable integer
- changed the data type in `user_score` column to float
- removed missing values in `name` and `genre` columns (same rows), as there were only two and sales for these games were close to 0M
- filled part of the gaps (21) in `year_of_release` using the `return_year` function that extracted the release year from the game name; left the remaining gaps unfilled
- analyzed the distribution of missing values in `user_score` and `critic_score` by release year, platform, and genre; found no ratings for games older than 2000 and on obsolete platforms, decided not to fill those gaps, filled the rest with median by genre
- replaced `K-A` in `rating` column with `E` according to ESRB categories
- removed implicit duplicates;
- added a column with total sales across all regions named `total_sales`

**Possible reasons for missing values:**
- data is genuinely missing: for irrelevant periods (before 2000s) when the industry was developing, for lesser-known games or platforms
- data was lost during collection or import and needs review and supplementation

[Back to navigation](#navigation)


<a id="step-3-exploratory-data-analysis"></a>
## Step 3 Exploratory Data Analysis

In [ ]:
data['year_of_release'].value_counts().sort_index()
# number of games released each year

In [ ]:
game_counts = data['year_of_release'].value_counts().sort_index()
game_counts.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title("Distribution of released games by release year")
plt.xlabel('Release year')
plt.ylabel('Number of released games')
plt.show()

We studied this chart earlier and concluded that the most relevant period for forecasting is 2006 to 2016 inclusive, since this period has the most game releases and represents the peak of industry development. Earlier data, especially before the 2000s, is outdated as it reflects the early stage of industry development and has very few released games.

<a id="sales-by-platform-and-their-dynamics"></a>
### Sales by Platform and Their Dynamics

In [ ]:
sales_by_platform = data.groupby('platform')['total_sales'].sum()
sales_by_platform.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of total sales by platform')
plt.xlabel('Platform')
plt.ylabel('Total sales')
plt.show()

In [ ]:
sales_by_platform.describe()
# use describe method to select platforms with the highest sales
# consider sales above the 3rd quartile as the highest


In [ ]:
top_platforms = list(sales_by_platform[sales_by_platform > 304].index)
# create a list of platforms with the highest sales

In [ ]:
pivot_data = (
    data
    .query('platform in @top_platforms')
    .pivot_table(index='year_of_release', columns='platform', values='total_sales', aggfunc='sum')
)
# create a pivot table for top platforms by sales with sales per platform per year

In [ ]:
pivot_data = pivot_data.fillna(0)
# replace missing values with 0, since missing means no sales

In [ ]:
plt.figure(figsize=(10, 6))
for platform in pivot_data.columns:
    plt.plot(pivot_data.index, pivot_data[platform], marker='o', label=platform)
plt.title('Annual sales of top platforms')
plt.xlabel('Release year')
plt.ylabel('Total sales (million copies)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

From the line chart of sales distribution for top platforms by year, the top 3 platforms by sales are: PS2, PS, Wii. Among the top-selling platforms, only PS4 continues to grow; the life cycle of the others had ended by 2016. This means newer platforms that have not yet entered the top 25% are gaining traction.

In [ ]:
data.query('platform == "DS"')['year_of_release'].sort_values().unique()
# examine 'year_of_release' values for DS platform

The value 1985 does appear to be an outlier, as all other games on this platform were released after 2004. Let's remove this value. 

In [ ]:
data = data.query('not (platform == "DS" and year_of_release <= 1985)').reset_index(drop=True)
# remove all DS rows with year less than or equal to 1985

In [ ]:
pivot_data.loc[1985, 'DS'] = 0 
# zero out DS sales for 1985 in pivot_data

Let's calculate the lifecycle for all platforms

In [ ]:
life_cycle = data.groupby('platform')['year_of_release'].agg(first_year='min', last_year='max')
life_cycle['duration'] = life_cycle['last_year'] - life_cycle['first_year']
life_cycle
# create and display a table with the "lifetime" duration of all platforms

In [ ]:
life_cycle['duration'].describe()
# describe the duration column using the describe method

The median of this dataset is 6 years, which was also visible on the top platforms chart. This means platforms typically last about 6 years on average.

<a id="selecting-relevant-data"></a>
### Selecting Relevant Data

Earlier, when analyzing missing values in critic_score and user_score, I selected the most relevant period (2006-2016) and identified obsolete platforms. Let's create a mask that includes relevant years and excludes obsolete platforms.

In [ ]:
mask = (~data['platform'].isin(old_platforms)
    & data['year_of_release'].isin(years))

In [ ]:
actual_data = data.loc[mask]
# create a new dataframe actual_data with relevant dates

<a id="selecting-potentially-profitable-platforms"></a>
### Selecting Potentially Profitable Platforms

For the relevant data, we will similarly analyze sales by platform and their dynamics

In [ ]:
sales_by_platform_act = actual_data.groupby('platform')['total_sales'].sum()
sales_by_platform_act.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by platform for relevant data')
plt.xlabel('Platform')
plt.ylabel('Total sales')
plt.show()

The chart shows platforms with total sales close to 0M. These may be unprofitable or new and gaining popularity; we need to study sales dynamics by year.

In [ ]:
pivot_data_act = actual_data.pivot_table(index='year_of_release', columns='platform', values='total_sales', aggfunc='sum').fillna(0)
# create a pivot table pivot_data_act with total sales per platform per year
pivot_data_act

Keep only columns where total sales in 2016 are not zero.

In [ ]:
pivot_data_act = pivot_data_act[pivot_data_act.columns[pivot_data_act.iloc[-1] != 0]]

In [ ]:
for platform in pivot_data_act.columns:
    plt.figure(figsize=(3, 3))
    plt.plot(pivot_data_act.index, pivot_data_act[platform], label=platform, color='#73A9AD')
    plt.title(platform)
    plt.xlabel('Release year')
    plt.ylabel('Total sales (million copies)')
    plt.grid(True)
    plt.show()
# build charts of total sales distribution by year for platforms that already have profit in 2016

The charts show that in 2015, XOne and PS4 had peak sales; in 2014, WiiU did. All platforms show a decline toward 2016, which may be because recent data has not been processed. The listed platforms (XOne, PS4, WiiU) likely have a chance to become major sellers in coming years. For other platforms, the decline has lasted several years and their lifecycle is coming to an end.

In [ ]:
actual_platforms = ['XOne', 'WiiU', 'PSV', 'PS4', '3DS']
# exclude platforms with sales close to 0
pivot_data_act = pivot_data_act.loc[:, pivot_data_act.columns.isin(actual_platforms)]
actual_data = actual_data.query("platform in @actual_platforms")
# update data in the tables

<a id="box-plot-of-global-sales-by-platform"></a>
### Box Plot of Global Sales by Platform

In [ ]:
actual_data.boxplot(
    column='total_sales',
    by='platform',
    figsize=(10, 6)
)
plt.title('Global game sales by platform')
plt.suptitle('')
plt.xlabel('Platform')
plt.ylabel('Total sales (million)')
plt.tight_layout()
plt.show()

In [ ]:
actual_data.boxplot(
    column='total_sales',
    by='platform',
    figsize=(10, 6)
)
plt.title('Global game sales by platform')
plt.suptitle('')
plt.xlabel('Platform')
plt.ylabel('Total sales (million)')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
# take a closer look at the lower part of the chart

The box plot shows the following parameters for platforms:
 - PS4 and XOne: median ≈0.21M, large IQR (≈0.05–0.70), many upper outliers.
 - Wii U: median ≈0.22M, moderate IQR (≈0.10–0.52).
 - 3DS: median ≈0.12M, moderate IQR (≈0.05–0.33).
 - PSV: median ≈0.045M, narrow IQR (≈0.015–0.13), many upper outliers.
All platforms have outliers, some exceeding 14M copies sold. This may be due to bestseller releases that gained unusually high popularity.

<a id="impact-of-reviews-on-sales-within-a-popular-platform"></a>
### Impact of Reviews on Sales Within a Popular Platform

For analysis we consider the PS4 platform, as it had the highest sales in 2015 (120M copies sold)

In [ ]:
(actual_data
    .query('platform == "PS4"')
    .plot
    .scatter(x='critic_score',
             y='total_sales',
             title='Impact of critic scores on PS4 sales (scatter plot)',
             xlabel='Critic score',
             ylabel='Sales (million copies)',
             alpha=0.5,
             color='#73A9AD',
             figsize=(10,6)
             )
 )
plt.show()
# build a scatter plot to assess the impact of critic reviews on PS4 platform sales

In [ ]:
actual_data.query('platform == "PS4"')['critic_score'].corr(actual_data.query('platform == "PS4"')['total_sales'])
# calculate the correlation coefficient for critic scores and total sales for PS4 platform

The chart shows a weak relationship between critic scores and total sales; there are points with very high critic scores (up to 90) but near-zero sales. However, high sales are typical only for games with ratings above 60. The correlation coefficient indicates a moderate relationship between critic scores and PS4 game sales.

In [ ]:
(actual_data
    .query('platform == "PS4"')
    .plot
    .scatter(x='user_score',
             y='total_sales',
             title='Impact of user scores on PS4 sales (scatter plot)',
             xlabel='User score',
             ylabel='Sales (million copies)',
             alpha=0.5,
             color='#73A9AD',
             figsize=(10,6)
             )
 )
plt.show()
# build a scatter plot to assess the impact of user reviews on PS4 platform sales

In [ ]:
# build a scatter plot to assess the impact of critic reviews on PS4 platform sales
actual_data.query('platform == "PS4"')['user_score'].corr(actual_data.query('platform == "PS4"')['total_sales'])

For user scores, the relationship is much weaker: high sales (over 4M copies) also occur with low ratings (3–5). Many games have high ratings (6–9) but low sales, similar to the critic scores scatter plot. The correlation coefficient also indicates a weak relationship.

**Conclusion:** Critic scores moderately influence game sales (correlation 0.35); user scores have weak or no influence (correlation -0.06)

<a id="correlation-of-conclusions-with-sales-on-other-platforms"></a>
### Correlation of Conclusions with Sales on Other Platforms

In [ ]:
(actual_data
    .groupby('platform')
    [['total_sales', 'critic_score', 'user_score']]
    .agg('corr')
    .reset_index()
    .rename(columns={'level_1':'metric'})
    .query('metric=="total_sales" and platform != "PS4"')
)
# examine correlation coefficients between user/critic scores and total sales for other platforms

**Conclusion:** For WiiU and XOne, the relationship between total sales and critic scores is also moderate; for 3DS and PSV the correlation is lower and weak. For WiiU, user score correlation equals critic score correlation (moderate). For 3DS and XOne, the user score relationship is weak; for PSV it is almost absent.

<a id="impact-of-genre-on-game-sales"></a>
### Impact of Genre on Game Sales

In [ ]:
actual_data['genre'].value_counts().plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of games by genre')
plt.xlabel('Genre')
plt.ylabel('Number of games')
plt.show()
# build the overall distribution of games by genre

The most popular genres are Action, Role-playing, and Adventure.

To better assess platform profitability, we calculate the mean sales per game for each genre by dividing total genre sales by the number of games in that genre.

In [ ]:
avg_total_sales = (actual_data.groupby('genre')['total_sales'].sum()/actual_data['genre'].value_counts())*1000
# express the result in thousands of copies


In [ ]:
avg_total_sales.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of mean total sales by genre')
plt.xlabel('Genre')
plt.ylabel('Mean total sales')
plt.show()

In [ ]:
popular_genres = ['Shooter', 'Platform', 'Racing']
actual_data.query('genre in @popular_genres').boxplot(
    column='total_sales',
    by='genre',
    figsize=(10, 6)
)
plt.title('Global game sales by platform')
plt.suptitle('')
plt.xlabel('Platform')
plt.ylabel('Total sales (million)')
plt.tight_layout()
plt.show()
# view boxplot for popular genres 

In [ ]:
popular_genres = ['Shooter', 'Platform', 'Racing']
actual_data.query('genre in @popular_genres').boxplot(
    column='total_sales',
    by='genre',
    figsize=(10, 6)
)
plt.title('Global game sales by platform')
plt.suptitle('')
plt.xlabel('Platform')
plt.ylabel('Total sales (million)')
plt.ylim(0, 5)
plt.tight_layout()
plt.show()
# zoom in on the plot by limiting y-axis

In [ ]:
unpopular_genres = ['Adventure', 'Strategy', 'Puzzle']
actual_data.query('genre in @unpopular_genres').boxplot(
    column='total_sales',
    by='genre',
    figsize=(10, 6)
)
plt.title('Global game sales by platform')
plt.suptitle('')
plt.xlabel('Platform')
plt.ylabel('Total sales (million)')
plt.tight_layout()
plt.show()
# view boxplot for unpopular genres

In [ ]:
unpopular_genres = ['Adventure', 'Strategy', 'Puzzle']
actual_data.query('genre in @unpopular_genres').boxplot(
    column='total_sales',
    by='genre',
    figsize=(10, 6)
)
plt.title('Global game sales by platform')
plt.suptitle('')
plt.xlabel('Platform')
plt.ylabel('Total sales (million)')
plt.ylim(0, 0.5)
plt.tight_layout()
plt.show()
# zoom in on the plot by limiting y-axis

**Conclusion:** The highest mean sales per game are for Shooter, Platform, and Racing; the lowest are for Adventure, Strategy, and Puzzle.

*The boxplot for popular genres shows the distribution of total global sales (million units):*
• Shooter  
  – Median ≈ 0.7  
  – IQR ≈ 0.2–2.1  
  – Whiskers up to ≈ 4.5, outliers up to ≈ 14–15  

• Platform  
  – Median ≈ 0.3  
  – IQR ≈ 0.1–0.9  
  – Whiskers up to ≈ 2.2, outliers up to ≈ 10–11  

• Racing  
  – Median ≈ 0.2  
  – IQR ≈ 0.1–0.6  
  – Whiskers up to ≈ 1.1, outliers up to ≈ 12  

Shooter leads in mid-range sales and spread; Platform and Racing have lower medians, but all genres have occasional hits with sales over 10 million.

*The boxplot for unpopular genres shows the distribution of total global sales (million units):*

Adventure  
– Median ≈ 0.03 million  
– IQR ≈ 0.01–0.08 million  
– Whiskers up to ≈ 0.16 million, outliers up to ≈ 0.5 million (a few up to ≈ 1.7 million)  

Puzzle  
– Median ≈ 0.10 million  
– IQR ≈ 0.03–0.18 million  
– Whiskers up to ≈ 0.32 million, outliers up to ≈ 0.48 million (some up to ≈ 1.8 million)  

Strategy  
– Median ≈ 0.07 million  
– IQR ≈ 0.03–0.14 million  
– Whiskers up to ≈ 0.34 million, outliers up to ≈ 0.39 million  

Puzzle has the highest median and spread; Strategy and Adventure have lower center-of-distribution sales, but all three genres occasionally have hits over 0.5 million copies.

<a id="exploratory-analysis-conclusion"></a>
### Exploratory Analysis Conclusion

1. I limited the analysis to 2006–2016: that is when most games were released and the industry was at its peak. Data before the 2000s appear outdated—too few releases for forecasting.

2. By total sales, PS2, PS, and Wii lead. But by 2016 only PS4 continues to gain momentum; the others have passed their peak. On average, platform lifecycle is about 6 years.

3. For the 2017 forecast, I filtered data for 2006–2016 and removed obsolete platforms. This keeps the sample relevant and sufficient.

4. In this slice, XOne, WiiU, PSV, PS4, and 3DS show stable growth and high sales volumes. I consider them the most promising for investment and game releases.

5. On the boxplot, the median sales of all relevant platforms are about 0.05–0.1 million copies, but there are outliers over 14 million—likely blockbusters with atypically high sales.

6. On PS4, the correlation between critic_score and sales is moderate: most hits have critic scores ≥ 60, but there are also nominally high scores with nearly zero sales. User_score hardly correlates with volume—games with both low and high ratings have both successful and failed sales.

7. On WiiU and XOne, the critic–sales correlation is also moderate; on 3DS and PSV it is weak or almost absent; from user reviews the link is even weaker or not noticeable.

8. By genre, Action, Shooter, Role-Playing, and Sports lead in total sales; Shooter, Platform, and Racing lead in average sales per game. Genres Adventure, Strategy, and Puzzle consistently rank as outsiders.

[Back to navigation](#navigation)


<a id="step-4-create-a-user-profile-for-each-region"></a>
## Step 4 Create a User Profile for Each Region

<a id="north-america-region"></a>
### North America Region

In [ ]:
platform_sales_ratio_na = round(data.groupby('platform')['na_sales'].sum()/data['na_sales'].sum(),2)

In [ ]:
platform_sales_ratio_na.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by platform for North America')
plt.xlabel('Platform')
plt.ylabel('Platform share of total sales in North America')
plt.show()

In [ ]:
platform_sales_ratio_na.sort_values(ascending=False).head()
# display top 5 popular platforms for North America

 The gap between the leader (X360) and second place (PS2) is small—just 1 percentage point (p.p.). From PS2 to Wii, the share drops by 2 p.p. (from 13% to 11%). From Wii to DS there is also a 2 p.p. decline (from 11% to 9%). DS and PS3 hold the same share (9%), so there is no difference between them.

Thus, fourth and fifth positions are on the same level; the main drop in sales share occurs in the transition from second to third place and from third to fourth.

In [ ]:
genre_sales_ratio_na = round(data.groupby('genre')['na_sales'].sum()/data['na_sales'].sum(), 2)

In [ ]:
genre_sales_ratio_na.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by genre for North America')
plt.xlabel('Platform')
plt.ylabel('Genre share of total sales in North America')
plt.show()

In [ ]:
genre_sales_ratio_na.sort_values(ascending=False).head()
# display top 5 popular genres for North America

Between 1st (20%) and 2nd (16%)—a 4 p.p. lead (Action sells 25% better than Sports).
Between 2nd and 3rd—3 p.p. (≈18% gap).
Between 3rd and 4th—another 3 p.p.
Between 4th and 5th—only 1 p.p. (≈11% gap).
In total, top 5 captures 68% of the market; other genres share 32%.

In [ ]:
data.groupby('rating')['na_sales'].sum()

Most sales come from games rated E (everyone 6+), M (mature 17+), and T (teen 13+), as these ratings cover many people—children, teenagers, and adults. "Extreme" groups (adults-only AO and early childhood) have very low popularity by comparison.

<a id="europe-region"></a>
### Europe Region

In [ ]:
platform_sales_ratio_eu = round(data.groupby('platform')['eu_sales'].sum()/data['eu_sales'].sum(),2)

In [ ]:
platform_sales_ratio_eu.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by platform for Europe')
plt.xlabel('Platform')
plt.ylabel('Platform share of total sales in Europe')
plt.show()

In [ ]:
platform_sales_ratio_eu.sort_values(ascending=False).head()
# display top 5 popular platforms for Europe

PS3 and PS2 share first place (both were in the North America top 5); X360 and Wii lag by 3 p.p. (also in the North America top 5); PS is another 2 p.p. behind (it was not in the North America top 5). Thus, we can compare the top 5 platforms in North America and Europe: the composition is similar, but North American users prefer X360 more, while in Europe PS3 is preferred (last in the North America top). In both regions PS2 is second with roughly the same share (13–14%).

In [ ]:
genre_sales_ratio_eu = round(data.groupby('genre')['eu_sales'].sum()/data['eu_sales'].sum(), 2)

In [ ]:
genre_sales_ratio_eu.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by genre for Europe')
plt.xlabel('Platform')
plt.ylabel('Genre share of total sales in Europe')
plt.show()

In [ ]:
genre_sales_ratio_eu.sort_values(ascending=False).head()
# display top 5 popular genres for Europe

The first 3 positions in the top 5 genres for Europe match North America—Action, Sports, Shooter—with nearly the same share distribution. Fourth place in Europe's top 5 is Racing; in North America with the same percentage it is Platform. Fifth place again matches (Misc, same percentage). This indicates similar genre preferences among users in North America and Europe.

In [ ]:
data.groupby('rating')['eu_sales'].sum()

For Europe, the ESRB rating distribution is the same: most sales are E, M, T. However, in Europe M-rated games outsell T-rated; in North America the opposite was observed. 

<a id="japan-region"></a>
### Japan Region

In [ ]:
platform_sales_ratio_jp = round(data.groupby('platform')['jp_sales'].sum()/data['jp_sales'].sum(),2)

In [ ]:
platform_sales_ratio_jp.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by platform for Japan')
plt.xlabel('Platform')
plt.ylabel('Platform share of total sales in Japan')
plt.show()

In [ ]:
platform_sales_ratio_jp.sort_values(ascending=False).head()
# display top 5 popular platforms for Japan

In Japan, the handheld DS ranks first (14%); PS and PS2 are 3 p.p. below (11% each, both in Europe top 5, but only PS2 in North America); SNES is 2 p.p. lower (9%, absent from Western tops); NES is 1 p.p. lower (8%, retro-region exclusive). 

In [ ]:
genre_sales_ratio_jp = round(data.groupby('genre')['jp_sales'].sum()/data['jp_sales'].sum(), 2)

In [ ]:
genre_sales_ratio_jp.plot(kind='bar', color='#73A9AD', figsize=(10,6))
plt.title('Distribution of sales by genre for Japan')
plt.xlabel('Platform')
plt.ylabel('Genre share of total sales in Japan')
plt.show()

In [ ]:
genre_sales_ratio_jp.sort_values(ascending=False).head()
# display top 5 popular genres for Japan

In Japan, Role-Playing firmly takes first place (27%; this genre is absent from US and Europe top 5); Action lags by 15 p.p. (12%; leader in North America at 20% and Europe at 21%); Platform and Sports are 2 p.p. lower (10% each: Platform 4th in US, outside Europe top 5; Sports 2nd in US and Europe at 16%); Misc is 2 p.p. behind (8%; Western markets 9% each). Shooter and Racing (13% and 10% in US/Europe) are not in Japan's top 5.  
Thus, Japanese players show a clear preference for role-playing games, while Action, Sports, and Shooter/Racing remain dominant in the Western hemisphere.

In [ ]:
data.groupby('rating')['jp_sales'].sum()

In Japan, E, M, and T ratings still lead. Teen significantly outweighs Mature (more than 2×), as in North America.

<a id="conclusion"></a>
### Conclusion
**North America User Profile**
Typical player:
⦁ Age: 12–35 years (children, teenagers, young adults)
⦁ Gender: predominantly male
⦁ Main platforms:  
  • X360 (14 %)  
  • PS2 (13 %)  
  • Wii (11 %)  
  • DS (9 %)  
  • PS3 (9 %)
⦁ Favorite genres:  
  • Action (20 %)  
  • Sports (16 %)  
  • Shooter (13 %)  
  • Platform (10 %)  
  • Misc (9 %)
⦁ ESRB ratings: most sales are E, T, M; Teen slightly leads Mature
⦁ Behavior: active in competitive shooters and sports simulators, value evening online sessions, interest in family (Wii) and handheld (DS) entertainment

 **European User Profile**
Typical player:
⦁ Age: 12–35 years
⦁ Gender: male and female (slightly more males)
⦁ Main platforms:  
  • PS3 and PS2 (≈14 % each)  
  • X360 and Wii (≈11 % each)  
  • PS (9 %)
⦁ Favorite genres:  
  • Action (21 %)  
  • Sports (16 %)  
  • Shooter (13 %)  
  • Racing (10 %)  
  • Misc (9 %)
⦁ ESRB ratings: E, M, T—main segments; Mature slightly exceeds Teen
⦁ Behavior: active in online competitions, like racing and sports games, interest in handheld

 **Japan User Profile**
Typical player:
⦁ Age: 10–30 years (many teenagers and young adults)
⦁ Gender: male and female
⦁ Main platforms:  
  • DS (14 %)  
  • PS and PS2 (11 % each)  
  • SNES (9 %)  
  • NES (8 %)
⦁ Favorite genres:  
  • Role-Playing (27 %)  
  • Action (12 %)  
  • Platform and Sports (10 % each)  
  • Misc (8 %)
⦁ ESRB ratings: E, T, M—main; Teen exceeds Mature by more than 2×
⦁ Behavior: passion for JRPG and handheld, strong nostalgia for retro consoles

North American and European gamers are very similar in tastes and devices—only platform priority and slight differences in age-rating emphasis differ.  
The Japanese market forms its own ecosystem: handheld, classic consoles, and JRPG dominance make it unique.

[Back to navigation](#navigation)


<a id="step-5-hypothesis-testing"></a>
## Step 5 Hypothesis Testing

<a id="mean-user-ratings-for-xbox-one-and-pc-platforms-are-equal"></a>
### Mean user ratings for Xbox One and PC platforms are equal

In [ ]:
data.query('platform=="XOne" and year_of_release in @years')['user_score'].mean()
# calculate mean user score for Xbox One

In [ ]:
data.query('platform=="PC" and year_of_release in @years')['user_score'].mean()
# calculate mean user score for PC platform

Formulate hypotheses:
- H0 — user ratings for Xbox One and PC platforms are equal;
- H1 — user ratings for Xbox One and PC platforms are not equal

In [ ]:
alpha = 0.05
# set significance level at 5%

In [ ]:
results = st.ttest_ind(
    data.query('platform=="XOne" and year_of_release in @years')['user_score'].dropna(),
    data.query('platform=="PC" and year_of_release in @years')['user_score'].dropna(),
    alternative='two-sided')
# run t-test for hypothesis, removing NaN values from samples first

In [ ]:
f'p-value: {results.pvalue}'
# output the p-value

In [ ]:
if results.pvalue < alpha:
    print('We reject the null hypothesis')
else:
    print('We cannot reject the null hypothesis')

**Conclusion:** We cannot claim that user ratings on Xbox One and PC platforms are equal. Note that the hypothesis would not have been rejected at alpha=0.01.

<a id="mean-user-ratings-for-action-and-sports-genres-are-different"></a>
### Mean user ratings for Action and Sports genres are different.

In [ ]:
data.query('genre=="Action" and year_of_release in @years')['user_score'].mean()
# calculate mean user score for Action genre

In [ ]:
data.query('genre=="Sports" and year_of_release in @years')['user_score'].mean()
# calculate mean user score for Sports genre

Formulate hypotheses:
- H0 — user ratings for Action and Sports genres are equal;
- H1 — user ratings for Action and Sports genres are not equal

In [ ]:
alpha = 0.05
# set significance level at 5%

In [ ]:
results = st.ttest_ind(
    data.query('genre=="Action" and year_of_release in @years')['user_score'].dropna(),
    data.query('genre=="Sports" and year_of_release in @years')['user_score'].dropna(),
    alternative='two-sided')
# run t-test for hypothesis, removing NaN values from samples first

In [ ]:
f'p-value: {results.pvalue}'
# output the p-value

In [ ]:
if results.pvalue < alpha:
    print('We reject the null hypothesis')
else:
    print('We cannot reject the null hypothesis')

**Conclusion:** We reject the null hypothesis of equal ratings for Action and Sports genres; we can therefore accept that these ratings differ.

[Back to navigation](#navigation)


<a id="overall-conclusion"></a>
## Overall Conclusion

<a id="summary-of-work-done"></a>
### Summary of Work Done

I completed the following steps in this project:

1. Data preparation
    I converted column names to snake_case and replaced "tbd" in user_score with NaN.
   ⦁ Changed year_of_release to nullable int, user_score and critic_score to float.
   ⦁ Removed 2 rows without name and genre (their sales ≈ 0).
   ⦁ Restored 21 release year from the name; left other gaps empty.
   ⦁ Analyzed gaps by year, platform, and genre: did not fill data for games before 2000 and obsolete platforms; filled the rest with genre median.
   ⦁ Mapped K-A rating to E and added total_sales column (sum of worldwide sales).

2. Exploratory analysis
   ⦁ Peak period: 2006–2016 (maximum releases and sales volume); data before 2000 is not representative.
   ⦁ Sales leaders: PS2, PS, Wii; by 2016 PS4, XOne, WiiU, PSV and 3DS are growing. Platform lifecycle ≈ 6 years.
   ⦁ In the current slice (2006–2016) the most promising platforms: PS4, Xbox One, WiiU, PlayStation Vita, Nintendo 3DS.
   ⦁ Box plot showed median sales 0.05–0.1 million copies, but there are blockbusters with > 14 million.

3. Impact of ratings
   ⦁ Moderate correlation between critic_score and sales (critic ratings matter more).
   ⦁ User_score has little effect on volume.

4. Genre analysis
   ⦁ Action, Shooter, Role-Playing, and Sports lead in total sales.
   ⦁ By average revenue per game: Shooter, Platform, Racing.
   ⦁ Outsiders: Adventure, Strategy, Puzzle.

5. User profiles
   ⦁ NA/EU: age 12–35, men and women; top 5 platforms X360, PS2, PS3, Wii, DS; genres Action, Sports, Shooter; ratings E/T/M.
   ⦁ JP: age 10–30, handheld and retro consoles (DS, PS, SNES, NES); genre #1 — JRPG; ratings E and T.

6. Hypothesis testing
   ⦁ Mean user_score for XOne and PC differ statistically (null hypothesis rejected).
   ⦁ Mean user_score for Action and Sports differ (null hypothesis rejected).

<a id="2017-forecast"></a>
### 2017 Forecast

In 2017, focus on PS4 and Xbox One, as they are gaining momentum and had high sales in 2015 (120 and 60 million copies). For promotion, choose Action and Shooter genres in North America and Europe, Role-playing in Japan; for casual audience consider Platform and Racing. Target group in North America and Europe: age 12–35, men and women, E/T/M rating, emphasis on competitive and co-op experience; in Japan—age 10–30, JRPG and handheld fans; interest in nostalgic and retro games, T and E ratings. In ad campaigns, mention high critic_score (above 60) as a quality guarantee.

2. Genres for promotion  
   – Action and Shooter (mass segment, high CTR);  
   – RPG (especially JRPG in JP);  
   – Sports (tournament and custom content for NA/EU);  
   – Additionally: Platform and Racing for casual audience.

3. Target groups  
   • North America and Europe:  
   – Age 12–35, men and women, E/T/M rating;  
   – Emphasis on competitive and co-op experience.  
   • Japan:  
   – Age 10–30, JRPG and handheld fans;  
   – Interest in nostalgic and retro games, T rating.

4. Key messages  
   – Emphasize high critic_score (≥ 60) as quality guarantee;  
   – Mention user_score second;  
   – Focus on online interaction, social features, and esports content.

Thus, by focusing on popular platforms and genres, targeting key audience segments, and collaborating with opinion leaders, we can maximize the effectiveness of ad campaigns in 2017.

[Back to navigation](#navigation)
